# Clinical Validation Suite — Glaucoma Detection Models

**Purpose**: Stress-test R28 (any glaucoma) and R32 (referable glaucoma) models
for clinical feasibility beyond standard ML metrics.

| Test | What It Answers | Clinical Relevance |
|------|----------------|-------------------|
| **1. Bootstrap CIs** | Are the F1/AUC estimates stable? | Regulators need confidence bounds |
| **2. Clinical Operating Points** | What happens at 90%/95% sensitivity? | Screening needs high sensitivity |
| **3. Subgroup Fairness** | Does it fail for specific demographics? | Equity in healthcare |
| **4. Calibration** | Can clinicians trust the probability? | "70% risk" must mean 70% |
| **5. Error Profiling** | Who gets misclassified? | Identify systematic blind spots |
| **6. Inter-Encoder Agreement** | Do models agree on difficult cases? | Ensemble reliability |
| **7. Threshold Stability** | How fragile is the optimal threshold? | Robustness to deployment drift |
| **8. Decision Curve Analysis** | Is there net clinical benefit? | Treat-all vs model-guided |
| **9. Clinical Readiness Summary** | Pass/fail checklist | Go/no-go decision |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 1 — Imports
# ═══════════════════════════════════════════════════════════════════════
from __future__ import annotations
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, balanced_accuracy_score,
    roc_curve, confusion_matrix,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

print('Clinical Validation Suite — imports OK')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 2 — Configuration + Load All Predictions & Metadata
# ═══════════════════════════════════════════════════════════════════════

BASE_DIR = Path.cwd()

# ─── Paths ────────────────────────────────────────────────────────
R28_DIR = BASE_DIR / 'results' / 'brset_r28_fe_experiment' / 'r28_per_encoder'
R32_DIR = BASE_DIR / 'results' / 'brset_r32_e2e_cascade_referable_glaucoma' / 'r32_per_encoder'
LABELS_PATH = BASE_DIR / 'data' / 'brset_embeddings' / 'brset_labels' / 'labels_brset.csv'
SPLITS_PATH = BASE_DIR / 'artifacts' / 'brset_eda_fe' / 'splits_patient.csv'
OUT_DIR = BASE_DIR / 'results' / 'clinical_validation'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ─── Load metadata ────────────────────────────────────────────────
labels = pd.read_csv(LABELS_PATH)
labels['patient_id'] = labels['patient_id'].astype(str)
labels = labels.rename(columns={'patient_age': 'age', 'patient_sex': 'sex'})

# Patient-level metadata (take first image per patient for demographics)
meta = labels.groupby('patient_id').first()[['age', 'sex', 'diabetes', 'camera']].copy()
meta['age_group'] = pd.cut(meta['age'], bins=[0, 40, 60, 120], labels=['<40', '40-59', '60+'])
meta['sex_label'] = meta['sex'].map({0: 'Female', 1: 'Male'})
meta['diabetes_label'] = meta['diabetes'].astype(str).str.lower().map({'yes': 'Diabetic', 'no': 'Non-diabetic'})
meta['camera_label'] = meta['camera']

# ─── Load splits ──────────────────────────────────────────────────
splits = pd.read_csv(SPLITS_PATH)
splits['patient_id'] = splits['patient_id'].astype(str)
test_pids = set(splits[splits['split'] == 'test']['patient_id'])

# ─── Load R28 predictions ─────────────────────────────────────────
def load_r28_predictions():
    """Load all R28 per-encoder patient predictions."""
    preds = {}
    if not R28_DIR.exists():
        print('  R28 results not found')
        return preds
    for enc_dir in sorted(R28_DIR.iterdir()):
        if not enc_dir.is_dir(): continue
        pred_file = enc_dir / 'patient_predictions_P25.csv'
        hp_file = enc_dir / 'hyperparameters.json'
        if not pred_file.exists(): continue
        df = pd.read_csv(pred_file)
        df['patient_id'] = df['patient_id'].astype(str)
        # Standardize columns
        df = df.rename(columns={'y_true': 'y_true', 'y_pred': 'y_pred', 'y_proba': 'y_proba'})
        enc_name = enc_dir.name.split('_', 1)[1] if '_' in enc_dir.name else enc_dir.name
        # Load threshold from hyperparameters
        threshold = 0.5
        if hp_file.exists():
            with open(hp_file) as f:
                hp = json.load(f)
                threshold = hp.get('threshold', 0.5)
        df['threshold'] = threshold
        preds[enc_name] = df
    return preds

# ─── Load R32 predictions ─────────────────────────────────────────
def load_r32_predictions():
    """Load all R32 per-encoder patient predictions."""
    preds = {}
    if not R32_DIR.exists():
        print('  R32 results not found')
        return preds
    for enc_dir in sorted(R32_DIR.iterdir()):
        if not enc_dir.is_dir(): continue
        pred_file = enc_dir / 'patient_predictions.csv'
        hp_file = enc_dir / 'hyperparameters.json'
        if not pred_file.exists(): continue
        df = pd.read_csv(pred_file)
        df['patient_id'] = df['patient_id'].astype(str)
        # Standardize columns to match R28 format
        df = df.rename(columns={
            'y_true_referable': 'y_true',
            'y_pred_referable': 'y_pred',
            'max_proba': 'y_proba',
        })
        enc_name = enc_dir.name.split('_', 1)[1] if '_' in enc_dir.name else enc_dir.name
        threshold = 0.5
        if hp_file.exists():
            with open(hp_file) as f:
                hp = json.load(f)
                threshold = hp.get('bilateral_threshold', 0.5)
        df['threshold'] = threshold
        preds[enc_name] = df
    return preds

r28_preds = load_r28_predictions()
r32_preds = load_r32_predictions()

# Build task dict
tasks = {}
if r28_preds:
    tasks['R28 Any Glaucoma'] = r28_preds
if r32_preds:
    tasks['R32 Referable Glaucoma'] = r32_preds

for task_name, preds in tasks.items():
    encs = list(preds.keys())
    n_patients = len(preds[encs[0]]) if encs else 0
    print(f'{task_name}: {len(encs)} encoders, {n_patients} test patients')
    for e in encs:
        prev = preds[e]['y_true'].mean()
        print(f'  {e}: prevalence={prev:.1%}, threshold={preds[e]["threshold"].iloc[0]:.3f}')

if not tasks:
    raise RuntimeError('No model predictions found! Run R28 or R32 first.')

print(f'\nMetadata: {len(meta)} patients, columns: {list(meta.columns)}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 3 — Test 1: Bootstrap Confidence Intervals
# ═══════════════════════════════════════════════════════════════════════
# Clinical requirement: metrics must have tight CIs for regulatory claims
# Rule of thumb: CI width < 0.08 is acceptable for a pilot study

N_BOOT = 2000
ALPHA = 0.05

def bootstrap_metrics(y_true, y_pred, y_proba, n_boot=N_BOOT, alpha=ALPHA, seed=42):
    """Bootstrap CIs for F1, Precision, Recall, AUC, Specificity, NPV, PPV."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    results = {m: [] for m in ['F1', 'Precision', 'Recall', 'AUC',
                                'Specificity', 'NPV', 'PPV', 'BalAcc']}
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt, yp, ypr = y_true[idx], y_pred[idx], y_proba[idx]
        if len(np.unique(yt)) < 2: continue
        cm = confusion_matrix(yt, yp)
        tn, fp, fn, tp = cm.ravel()
        results['F1'].append(f1_score(yt, yp, zero_division=0))
        results['Precision'].append(precision_score(yt, yp, zero_division=0))
        results['Recall'].append(recall_score(yt, yp, zero_division=0))
        results['AUC'].append(roc_auc_score(yt, ypr))
        results['Specificity'].append(tn / (tn + fp) if (tn + fp) > 0 else 0)
        results['NPV'].append(tn / (tn + fn) if (tn + fn) > 0 else 0)
        results['PPV'].append(tp / (tp + fp) if (tp + fp) > 0 else 0)
        results['BalAcc'].append(balanced_accuracy_score(yt, yp))
    lo = alpha / 2
    hi = 1 - alpha / 2
    summary = {}
    for m, vals in results.items():
        vals = np.array(vals)
        summary[m] = {
            'mean': np.mean(vals),
            'lo': np.percentile(vals, lo * 100),
            'hi': np.percentile(vals, hi * 100),
            'width': np.percentile(vals, hi * 100) - np.percentile(vals, lo * 100),
        }
    return summary

print('═' * 80)
print('TEST 1: Bootstrap Confidence Intervals (2000 resamples, 95% CI)')
print('═' * 80)

all_ci = []
for task_name, preds in tasks.items():
    print(f'\n── {task_name} ──')
    for enc_name, df in preds.items():
        ci = bootstrap_metrics(
            df['y_true'].values, df['y_pred'].values, df['y_proba'].values
        )
        row = {'task': task_name, 'encoder': enc_name}
        for m, stats in ci.items():
            row[f'{m}_mean'] = stats['mean']
            row[f'{m}_lo'] = stats['lo']
            row[f'{m}_hi'] = stats['hi']
            row[f'{m}_width'] = stats['width']
        all_ci.append(row)
        print(f'  {enc_name}:')
        for m in ['F1', 'AUC', 'Recall', 'Specificity', 'NPV']:
            s = ci[m]
            flag = ' ⚠' if s['width'] > 0.08 else ' ✓'
            print(f'    {m:>12s}: {s["mean"]:.3f} [{s["lo"]:.3f}, {s["hi"]:.3f}]  '
                  f'(width={s["width"]:.3f}){flag}')

ci_df = pd.DataFrame(all_ci)
ci_df.to_csv(OUT_DIR / 'test1_bootstrap_ci.csv', index=False)

# ─── Forest plot ──────────────────────────────────────────────────
for task_name, preds in tasks.items():
    enc_names = list(preds.keys())
    fig, axes = plt.subplots(1, 5, figsize=(20, max(3, len(enc_names) * 0.55)),
                             sharey=True)
    metrics_to_plot = ['F1', 'AUC', 'Recall', 'Specificity', 'NPV']
    task_ci = ci_df[ci_df['task'] == task_name]

    for ax, metric in zip(axes, metrics_to_plot):
        for i, enc in enumerate(enc_names):
            row = task_ci[task_ci['encoder'] == enc].iloc[0]
            mean = row[f'{metric}_mean']
            lo = row[f'{metric}_lo']
            hi = row[f'{metric}_hi']
            color = '#d32f2f' if (hi - lo) > 0.08 else '#1976d2'
            ax.errorbar(mean, i, xerr=[[mean-lo], [hi-mean]], fmt='o',
                       color=color, capsize=4, markersize=6, linewidth=1.5)
        ax.set_title(metric, fontsize=11, fontweight='bold')
        ax.axvline(0.80, color='#666', ls='--', lw=0.8, alpha=0.5)
        ax.set_xlim(0.55, 1.02)
        ax.set_xlabel('Score')

    axes[0].set_yticks(range(len(enc_names)))
    axes[0].set_yticklabels(enc_names, fontsize=9)
    fig.suptitle(f'{task_name} — Bootstrap 95% CI (blue=tight, red=wide)',
                fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    fig.savefig(OUT_DIR / f'test1_ci_{task_name.split()[0].lower()}.png',
               dpi=160, bbox_inches='tight')
    plt.show()

print('\n✓ Test 1 complete — CIs saved')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 4 — Test 2: Clinical Operating Points
# ═══════════════════════════════════════════════════════════════════════
# Clinicians don't use F1 — they ask:
#   "At 90% sensitivity, how many false positives do I get?"
#   "At 95% sensitivity, what's my specificity?"
#   "If the model says negative, how safe is that?" (NPV)

SENS_TARGETS = [0.90, 0.95, 0.80]  # screening, high-safety, balanced

def clinical_operating_points(y_true, y_proba, sens_targets=SENS_TARGETS):
    """Find threshold that achieves target sensitivity, report all metrics there."""
    fpr, tpr, thresholds = roc_curve(y_true, y_proba)
    results = []
    for target_sens in sens_targets:
        # Find threshold closest to target sensitivity
        valid = tpr >= target_sens
        if not valid.any():
            results.append({'target_sens': target_sens, 'achievable': False})
            continue
        # Among all thresholds achieving >= target, pick the one with lowest FPR
        idx = np.where(valid)[0]
        best_idx = idx[np.argmin(fpr[idx])]
        thr = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
        actual_sens = tpr[best_idx]
        actual_spec = 1 - fpr[best_idx]

        y_pred_at_thr = (y_proba >= thr).astype(int)
        cm = confusion_matrix(y_true, y_pred_at_thr)
        tn, fp, fn, tp = cm.ravel()
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
        f1 = f1_score(y_true, y_pred_at_thr, zero_division=0)

        results.append({
            'target_sens': target_sens,
            'achievable': True,
            'threshold': float(thr),
            'sensitivity': float(actual_sens),
            'specificity': float(actual_spec),
            'PPV': float(ppv),
            'NPV': float(npv),
            'F1': float(f1),
            'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
            'false_referral_rate': float(fp / (fp + tn)) if (fp + tn) > 0 else 0,
        })
    return results

print('═' * 80)
print('TEST 2: Clinical Operating Points')
print('═' * 80)
print('Question: At clinically meaningful sensitivity levels, what trade-offs exist?\n')

all_ops = []
for task_name, preds in tasks.items():
    print(f'\n── {task_name} ──')
    for enc_name, df in preds.items():
        ops = clinical_operating_points(df['y_true'].values, df['y_proba'].values)
        print(f'\n  {enc_name}:')
        for op in ops:
            if not op['achievable']:
                print(f'    Sens≥{op["target_sens"]:.0%}: NOT ACHIEVABLE')
                continue
            npv_flag = ' ✓' if op['NPV'] >= 0.97 else ' ⚠ <97%'
            print(f'    Sens≥{op["target_sens"]:.0%}: thr={op["threshold"]:.3f}  '
                  f'Sens={op["sensitivity"]:.1%}  Spec={op["specificity"]:.1%}  '
                  f'PPV={op["PPV"]:.1%}  NPV={op["NPV"]:.1%}{npv_flag}  '
                  f'F1={op["F1"]:.3f}  FalseRef={op["false_referral_rate"]:.1%}')
            op.update({'task': task_name, 'encoder': enc_name})
            all_ops.append(op)

ops_df = pd.DataFrame(all_ops)
ops_df.to_csv(OUT_DIR / 'test2_clinical_operating_points.csv', index=False)

# ─── Visualization: Sens vs Spec trade-off per encoder ─────────
for task_name, preds in tasks.items():
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = plt.cm.tab10(np.linspace(0, 1, len(preds)))
    for (enc_name, df), color in zip(preds.items(), colors):
        fpr, tpr, _ = roc_curve(df['y_true'].values, df['y_proba'].values)
        ax.plot(1 - fpr, tpr, label=enc_name, color=color, lw=1.8)

    # Add clinical targets
    ax.axhline(0.90, color='red', ls='--', lw=0.8, alpha=0.6, label='90% Sens')
    ax.axhline(0.95, color='darkred', ls=':', lw=0.8, alpha=0.6, label='95% Sens')
    ax.axvline(0.80, color='blue', ls='--', lw=0.8, alpha=0.4, label='80% Spec')
    ax.set_xlabel('Specificity', fontsize=11)
    ax.set_ylabel('Sensitivity', fontsize=11)
    ax.set_title(f'{task_name} — Sensitivity vs Specificity\n'
                 f'(dashed lines = clinical targets)', fontsize=12)
    ax.legend(fontsize=8, loc='lower left')
    ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.02)
    plt.tight_layout()
    fig.savefig(OUT_DIR / f'test2_sens_spec_{task_name.split()[0].lower()}.png',
               dpi=160, bbox_inches='tight')
    plt.show()

print('\n✓ Test 2 complete — operating points saved')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 5 — Test 3: Subgroup Fairness Analysis
# ═══════════════════════════════════════════════════════════════════════
# Regulatory requirement: model must not systematically fail for
# protected groups. FDA & ANVISA require demographic breakdowns.
#
# PASS criteria:
#   - No subgroup F1 drops > 0.10 below overall F1
#   - No subgroup has Recall < 0.70 (missed cases)
#   - No subgroup has NPV < 0.90 (unsafe "all clear")

SUBGROUPS = {
    'Age': 'age_group',
    'Sex': 'sex_label',
    'Diabetes': 'diabetes_label',
    'Camera': 'camera_label',
}

FAIRNESS_F1_DROP = 0.10
FAIRNESS_MIN_RECALL = 0.70
FAIRNESS_MIN_NPV = 0.90

print('═' * 80)
print('TEST 3: Subgroup Fairness Analysis')
print('═' * 80)
print(f'PASS criteria: F1 drop < {FAIRNESS_F1_DROP}, Recall ≥ {FAIRNESS_MIN_RECALL}, NPV ≥ {FAIRNESS_MIN_NPV}\n')

all_fairness = []
fairness_flags = []

for task_name, preds in tasks.items():
    print(f'\n── {task_name} ──')
    for enc_name, df in preds.items():
        # Merge with metadata
        merged = df.merge(meta, on='patient_id', how='left')
        overall_f1 = f1_score(merged['y_true'], merged['y_pred'], zero_division=0)

        print(f'\n  {enc_name} (overall F1={overall_f1:.3f}):')

        for group_label, group_col in SUBGROUPS.items():
            if group_col not in merged.columns:
                continue
            for val, sub in merged.groupby(group_col):
                if len(sub) < 20: continue  # skip tiny groups
                n = len(sub)
                n_pos = sub['y_true'].sum()
                prev = n_pos / n if n > 0 else 0

                yt, yp = sub['y_true'].values, sub['y_pred'].values
                sub_f1 = f1_score(yt, yp, zero_division=0)
                sub_rec = recall_score(yt, yp, zero_division=0) if n_pos > 0 else np.nan
                sub_prec = precision_score(yt, yp, zero_division=0) if yp.sum() > 0 else np.nan

                cm = confusion_matrix(yt, yp, labels=[0, 1])
                tn = cm[0, 0]; fn = cm[1, 0]
                sub_npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan

                f1_drop = overall_f1 - sub_f1
                flags = []
                if f1_drop > FAIRNESS_F1_DROP:
                    flags.append(f'F1 drop={f1_drop:+.3f}')
                if not np.isnan(sub_rec) and sub_rec < FAIRNESS_MIN_RECALL:
                    flags.append(f'Recall={sub_rec:.3f}')
                if not np.isnan(sub_npv) and sub_npv < FAIRNESS_MIN_NPV:
                    flags.append(f'NPV={sub_npv:.3f}')
                flag_str = ' ⚠ ' + ', '.join(flags) if flags else ' ✓'

                row = {
                    'task': task_name, 'encoder': enc_name,
                    'group': group_label, 'value': str(val),
                    'n': n, 'n_pos': int(n_pos), 'prevalence': prev,
                    'F1': sub_f1, 'Recall': sub_rec, 'Precision': sub_prec,
                    'NPV': sub_npv, 'F1_drop': f1_drop,
                }
                all_fairness.append(row)
                if flags:
                    fairness_flags.append(row)

                print(f'    {group_label}={val:>15s}  n={n:>5d} (pos={n_pos:>3d}, '
                      f'prev={prev:.1%})  F1={sub_f1:.3f}  Rec={sub_rec:.3f}  '
                      f'NPV={sub_npv:.3f}{flag_str}')

fairness_df = pd.DataFrame(all_fairness)
fairness_df.to_csv(OUT_DIR / 'test3_subgroup_fairness.csv', index=False)

# ─── Summary of failures ──────────────────────────────────────────
if fairness_flags:
    print(f'\n⚠ {len(fairness_flags)} subgroup violations found:')
    for f in fairness_flags:
        print(f'  {f["task"]} / {f["encoder"]} / {f["group"]}={f["value"]}: '
              f'F1={f["F1"]:.3f} (drop={f["F1_drop"]:+.3f}), '
              f'Recall={f["Recall"]:.3f}, NPV={f["NPV"]:.3f}')
else:
    print('\n✓ All subgroups PASS fairness criteria')

# ─── Heatmap ──────────────────────────────────────────────────────
for task_name, preds in tasks.items():
    task_fair = fairness_df[fairness_df['task'] == task_name].copy()
    if task_fair.empty: continue
    task_fair['label'] = task_fair['group'] + '=' + task_fair['value']
    enc_names = list(preds.keys())
    labels_order = task_fair.groupby('label')['F1'].mean().sort_values().index.tolist()

    heat = task_fair.pivot_table(index='label', columns='encoder', values='F1')
    heat = heat.reindex(index=labels_order, columns=enc_names)

    fig, ax = plt.subplots(figsize=(max(8, len(enc_names) * 1.5), max(4, len(labels_order) * 0.5)))
    im = ax.imshow(heat.values, cmap='RdYlGn', vmin=0.55, vmax=0.95, aspect='auto')
    ax.set_xticks(range(len(enc_names)))
    ax.set_xticklabels(enc_names, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(labels_order)))
    ax.set_yticklabels(labels_order, fontsize=9)
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            v = heat.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                       fontsize=8, fontweight='bold',
                       color='white' if v < 0.70 else 'black')
    fig.colorbar(im, ax=ax, fraction=0.03, pad=0.04, label='F1')
    ax.set_title(f'{task_name} — Subgroup F1 Heatmap', fontsize=12, fontweight='bold')
    plt.tight_layout()
    fig.savefig(OUT_DIR / f'test3_fairness_{task_name.split()[0].lower()}.png',
               dpi=160, bbox_inches='tight')
    plt.show()

print('\n✓ Test 3 complete — fairness results saved')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 6 — Test 4: Calibration Analysis
# ═══════════════════════════════════════════════════════════════════════
# A well-calibrated model: when it says "70% risk", ~70% truly have it.
# Critical for clinical decision-making and patient communication.
#
# Metrics: ECE (Expected Calibration Error), Brier Score, reliability diagram
# PASS: ECE < 0.05, Brier < 0.15

ECE_THRESHOLD = 0.05
BRIER_THRESHOLD = 0.15
N_CAL_BINS = 10

def expected_calibration_error(y_true, y_proba, n_bins=N_CAL_BINS):
    """Compute ECE: weighted average of |accuracy - confidence| per bin."""
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_data = []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (y_proba >= lo) & (y_proba < hi)
        if hi == 1.0:
            mask = mask | (y_proba == 1.0)
        n_in_bin = mask.sum()
        if n_in_bin == 0:
            bin_data.append((lo, hi, 0, 0, 0))
            continue
        avg_conf = y_proba[mask].mean()
        avg_acc = y_true[mask].mean()
        ece += (n_in_bin / len(y_true)) * abs(avg_acc - avg_conf)
        bin_data.append((lo, hi, n_in_bin, avg_conf, avg_acc))
    return ece, bin_data

print('═' * 80)
print('TEST 4: Calibration Analysis')
print('═' * 80)
print(f'PASS: ECE < {ECE_THRESHOLD}, Brier < {BRIER_THRESHOLD}\n')

cal_results = []

for task_name, preds in tasks.items():
    n_enc = len(preds)
    fig, axes = plt.subplots(1, n_enc, figsize=(4.5 * n_enc, 4.5), squeeze=False)
    axes = axes[0]

    print(f'\n── {task_name} ──')
    for idx, (enc_name, df) in enumerate(preds.items()):
        yt = df['y_true'].values
        yp = df['y_proba'].values

        brier = brier_score_loss(yt, yp)
        ece, bin_data = expected_calibration_error(yt, yp)

        ece_flag = ' ✓' if ece < ECE_THRESHOLD else ' ⚠ FAIL'
        brier_flag = ' ✓' if brier < BRIER_THRESHOLD else ' ⚠ FAIL'

        print(f'  {enc_name}: ECE={ece:.4f}{ece_flag}  Brier={brier:.4f}{brier_flag}')
        cal_results.append({'task': task_name, 'encoder': enc_name,
                           'ECE': ece, 'Brier': brier,
                           'ECE_pass': ece < ECE_THRESHOLD,
                           'Brier_pass': brier < BRIER_THRESHOLD})

        # Reliability diagram
        ax = axes[idx]
        try:
            frac_pos, mean_pred = calibration_curve(yt, yp, n_bins=N_CAL_BINS, strategy='uniform')
            ax.plot(mean_pred, frac_pos, 's-', color='#1976d2', lw=2, markersize=6, label='Model')
        except Exception:
            pass
        ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect')
        ax.fill_between([0, 1], [0, 1], alpha=0.05, color='green')
        ax.set_xlabel('Mean predicted probability')
        ax.set_ylabel('Fraction of positives')
        ax.set_title(f'{enc_name}\nECE={ece:.3f}, Brier={brier:.3f}', fontsize=9)
        ax.legend(fontsize=8)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)

    fig.suptitle(f'{task_name} — Reliability Diagrams', fontsize=13, fontweight='bold', y=1.03)
    plt.tight_layout()
    fig.savefig(OUT_DIR / f'test4_calibration_{task_name.split()[0].lower()}.png',
               dpi=160, bbox_inches='tight')
    plt.show()

pd.DataFrame(cal_results).to_csv(OUT_DIR / 'test4_calibration.csv', index=False)
print('\n✓ Test 4 complete — calibration results saved')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 7 — Test 5: Error Profiling
# ═══════════════════════════════════════════════════════════════════════
# Who gets misclassified? Are there demographic patterns in errors?
# This reveals systematic blind spots.

print('═' * 80)
print('TEST 5: Error Profiling')
print('═' * 80)

for task_name, preds in tasks.items():
    print(f'\n── {task_name} ──')
    # Use the best encoder for detailed error profiling
    best_enc = max(preds.keys(),
                   key=lambda e: f1_score(preds[e]['y_true'], preds[e]['y_pred'], zero_division=0))
    print(f'  (profiling best encoder: {best_enc})')
    df = preds[best_enc].merge(meta, on='patient_id', how='left')

    df['outcome'] = 'TN'
    df.loc[(df['y_true'] == 1) & (df['y_pred'] == 1), 'outcome'] = 'TP'
    df.loc[(df['y_true'] == 1) & (df['y_pred'] == 0), 'outcome'] = 'FN'
    df.loc[(df['y_true'] == 0) & (df['y_pred'] == 1), 'outcome'] = 'FP'

    print(f'\n  Outcome distribution:')
    for oc in ['TP', 'FP', 'FN', 'TN']:
        n = (df['outcome'] == oc).sum()
        pct = n / len(df) * 100
        print(f'    {oc}: {n:>5d} ({pct:.1f}%)')

    # Compare demographics of FN vs TP (missed positives vs caught positives)
    fn_df = df[df['outcome'] == 'FN']
    tp_df = df[df['outcome'] == 'TP']
    fp_df = df[df['outcome'] == 'FP']
    tn_df = df[df['outcome'] == 'TN']

    print(f'\n  False Negatives (missed cases) vs True Positives (caught cases):')
    print(f'    FN count: {len(fn_df)}, TP count: {len(tp_df)}')

    for feat, col in [('Age', 'age'), ('Sex', 'sex_label'),
                       ('Diabetes', 'diabetes_label'), ('Camera', 'camera_label')]:
        if col == 'age':
            fn_val = fn_df[col].median() if len(fn_df) > 0 else 0
            tp_val = tp_df[col].median() if len(tp_df) > 0 else 0
            print(f'    {feat}: FN median={fn_val:.0f}, TP median={tp_val:.0f}')
        else:
            if len(fn_df) > 0:
                fn_dist = fn_df[col].value_counts(normalize=True)
            else:
                fn_dist = pd.Series(dtype=float)
            if len(tp_df) > 0:
                tp_dist = tp_df[col].value_counts(normalize=True)
            else:
                tp_dist = pd.Series(dtype=float)
            print(f'    {feat}:')
            all_vals = set(fn_dist.index) | set(tp_dist.index)
            for v in sorted(all_vals):
                fn_pct = fn_dist.get(v, 0) * 100
                tp_pct = tp_dist.get(v, 0) * 100
                delta = fn_pct - tp_pct
                flag = ' ⚠' if abs(delta) > 15 else ''
                print(f'      {v}: FN={fn_pct:.1f}% vs TP={tp_pct:.1f}% (Δ={delta:+.1f}%){flag}')

    # ── Probability distribution by outcome ──
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    # Panel 1: probability histograms by outcome
    ax = axes[0]
    for oc, color, alpha in [('TP', '#2e7d32', 0.6), ('FN', '#d32f2f', 0.7),
                              ('FP', '#f57c00', 0.5), ('TN', '#1565c0', 0.3)]:
        subset = df[df['outcome'] == oc]['y_proba']
        if len(subset) > 0:
            ax.hist(subset, bins=30, alpha=alpha, label=f'{oc} (n={len(subset)})',
                   color=color, density=True)
    ax.axvline(df['threshold'].iloc[0], color='black', ls='--', lw=1.5, label=f'Threshold')
    ax.set_xlabel('Predicted Probability')
    ax.set_ylabel('Density')
    ax.set_title(f'{best_enc} — Probability by Outcome')
    ax.legend(fontsize=8)

    # Panel 2: age distribution by outcome
    ax = axes[1]
    for oc, color in [('TP', '#2e7d32'), ('FN', '#d32f2f'),
                       ('FP', '#f57c00'), ('TN', '#1565c0')]:
        subset = df[df['outcome'] == oc]['age'].dropna()
        if len(subset) > 5:
            ax.hist(subset, bins=20, alpha=0.5, label=f'{oc} (n={len(subset)})',
                   color=color, density=True)
    ax.set_xlabel('Age')
    ax.set_ylabel('Density')
    ax.set_title(f'{best_enc} — Age Distribution by Outcome')
    ax.legend(fontsize=8)

    fig.suptitle(f'{task_name} — Error Profiling', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    fig.savefig(OUT_DIR / f'test5_error_profile_{task_name.split()[0].lower()}.png',
               dpi=160, bbox_inches='tight')
    plt.show()

    # Save hard cases (FN with high proba — the model was close but missed)
    fn_near_miss = fn_df.nlargest(min(20, len(fn_df)), 'y_proba')
    fn_near_miss[['patient_id', 'y_true', 'y_pred', 'y_proba',
                  'age', 'sex_label', 'diabetes_label', 'camera_label']].to_csv(
        OUT_DIR / f'test5_fn_near_miss_{task_name.split()[0].lower()}.csv', index=False)

print('\n✓ Test 5 complete — error profiles saved')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 8 — Test 6: Inter-Encoder Agreement + Test 7: Threshold Stability
# ═══════════════════════════════════════════════════════════════════════

from itertools import combinations

def cohen_kappa(y1, y2):
    """Cohen's kappa for two binary prediction vectors."""
    cm = confusion_matrix(y1, y2, labels=[0, 1])
    n = cm.sum()
    po = np.diag(cm).sum() / n  # observed agreement
    pe = (cm.sum(axis=0) * cm.sum(axis=1)).sum() / (n * n)  # expected
    return (po - pe) / (1 - pe) if (1 - pe) > 0 else 0

print('═' * 80)
print('TEST 6: Inter-Encoder Agreement (Cohen\'s Kappa)')
print('═' * 80)
print('κ interpretation: <0.40 poor, 0.40-0.60 moderate, 0.60-0.80 substantial, >0.80 excellent\n')

all_kappa = []
for task_name, preds in tasks.items():
    enc_names = list(preds.keys())
    if len(enc_names) < 2:
        print(f'{task_name}: need ≥2 encoders for agreement analysis')
        continue

    print(f'\n── {task_name} ──')
    k_matrix = np.zeros((len(enc_names), len(enc_names)))

    for i, j in combinations(range(len(enc_names)), 2):
        e1, e2 = enc_names[i], enc_names[j]
        # Align by patient_id
        m = preds[e1][['patient_id', 'y_pred']].merge(
            preds[e2][['patient_id', 'y_pred']], on='patient_id', suffixes=('_1', '_2'))
        k = cohen_kappa(m['y_pred_1'].values, m['y_pred_2'].values)
        k_matrix[i, j] = k
        k_matrix[j, i] = k
        all_kappa.append({'task': task_name, 'enc1': e1, 'enc2': e2, 'kappa': k})
    np.fill_diagonal(k_matrix, 1.0)

    # Print matrix
    short = [e[:12] for e in enc_names]
    print(f'  {"":>14s}  ' + '  '.join(f'{s:>12s}' for s in short))
    for i, name in enumerate(short):
        vals = '  '.join(f'{k_matrix[i,j]:>12.3f}' for j in range(len(enc_names)))
        print(f'  {name:>14s}  {vals}')

    mean_k = np.mean([r['kappa'] for r in all_kappa if r['task'] == task_name])
    interp = 'poor' if mean_k < 0.4 else 'moderate' if mean_k < 0.6 else 'substantial' if mean_k < 0.8 else 'excellent'
    print(f'\n  Mean κ = {mean_k:.3f} ({interp})')

    # Heatmap
    fig, ax = plt.subplots(figsize=(max(6, len(enc_names) * 0.9), max(5, len(enc_names) * 0.8)))
    im = ax.imshow(k_matrix, cmap='YlOrRd', vmin=0.3, vmax=1.0)
    ax.set_xticks(range(len(enc_names)))
    ax.set_xticklabels(short, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(enc_names)))
    ax.set_yticklabels(short, fontsize=8)
    for i in range(len(enc_names)):
        for j in range(len(enc_names)):
            ax.text(j, i, f'{k_matrix[i,j]:.2f}', ha='center', va='center', fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Cohen's κ")
    ax.set_title(f'{task_name} — Inter-Encoder Agreement', fontsize=12, fontweight='bold')
    plt.tight_layout()
    fig.savefig(OUT_DIR / f'test6_kappa_{task_name.split()[0].lower()}.png',
               dpi=160, bbox_inches='tight')
    plt.show()

pd.DataFrame(all_kappa).to_csv(OUT_DIR / 'test6_inter_encoder_kappa.csv', index=False)

# ═══════════════════════════════════════════════════════════════════════
# Test 7: Threshold Stability
# ═══════════════════════════════════════════════════════════════════════
print(f'\n{"═" * 80}')
print('TEST 7: Threshold Stability (±5% perturbation)')
print('═' * 80)
print('Question: If the optimal threshold shifts slightly in deployment, does F1 collapse?\n')

PERTURB_RANGE = np.linspace(-0.05, 0.05, 21)  # ±5% around optimal

for task_name, preds in tasks.items():
    print(f'\n── {task_name} ──')
    fig, ax = plt.subplots(figsize=(8, 4.5))
    colors = plt.cm.tab10(np.linspace(0, 1, len(preds)))

    for (enc_name, df), color in zip(preds.items(), colors):
        base_thr = df['threshold'].iloc[0]
        yt = df['y_true'].values
        yp = df['y_proba'].values

        f1s = []
        for delta in PERTURB_RANGE:
            thr = np.clip(base_thr + delta, 0.01, 0.99)
            pred = (yp >= thr).astype(int)
            f1s.append(f1_score(yt, pred, zero_division=0))

        f1s = np.array(f1s)
        f1_at_opt = f1s[len(PERTURB_RANGE) // 2]
        worst_f1 = f1s.min()
        drop = f1_at_opt - worst_f1

        flag = ' ⚠ FRAGILE' if drop > 0.05 else ' ✓ stable'
        print(f'  {enc_name}: F1@opt={f1_at_opt:.3f}, worst±5%={worst_f1:.3f}, '
              f'max_drop={drop:.3f}{flag}')

        ax.plot(PERTURB_RANGE, f1s, '-o', color=color, markersize=3,
               label=f'{enc_name} (Δ={drop:.3f})', lw=1.5)

    ax.axvline(0, color='black', ls='--', lw=0.8)
    ax.axhspan(0, 0.70, color='red', alpha=0.05)
    ax.set_xlabel('Threshold Perturbation')
    ax.set_ylabel('F1 Score')
    ax.set_title(f'{task_name} — Threshold Stability (±5%)', fontsize=12, fontweight='bold')
    ax.legend(fontsize=7, loc='lower left')
    plt.tight_layout()
    fig.savefig(OUT_DIR / f'test7_stability_{task_name.split()[0].lower()}.png',
               dpi=160, bbox_inches='tight')
    plt.show()

print('\n✓ Tests 6 & 7 complete')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 9 — Test 8: Decision Curve Analysis (Net Benefit)
# ═══════════════════════════════════════════════════════════════════════
# DCA answers: "Over what range of threshold probabilities does the model
# provide more benefit than treat-all or treat-none?"
#
# Net Benefit = (TP/n) - (FP/n) × (pt / (1-pt))
# where pt = threshold probability (clinician's risk tolerance)

def decision_curve(y_true, y_proba, thresholds=np.linspace(0.01, 0.99, 200)):
    """Compute net benefit at each threshold for DCA."""
    n = len(y_true)
    nb_model = []
    nb_all = []
    for pt in thresholds:
        y_pred = (y_proba >= pt).astype(int)
        tp = ((y_pred == 1) & (y_true == 1)).sum()
        fp = ((y_pred == 1) & (y_true == 0)).sum()
        nb = (tp / n) - (fp / n) * (pt / (1 - pt)) if pt < 1 else 0
        nb_model.append(nb)
        # Treat-all
        prev = y_true.mean()
        nb_all.append(prev - (1 - prev) * (pt / (1 - pt)) if pt < 1 else 0)
    return thresholds, np.array(nb_model), np.array(nb_all)

print('═' * 80)
print('TEST 8: Decision Curve Analysis (Net Benefit)')
print('═' * 80)
print('Shows: range of threshold probs where model beats treat-all and treat-none.\n')

for task_name, preds in tasks.items():
    fig, ax = plt.subplots(figsize=(9, 5.5))
    colors = plt.cm.tab10(np.linspace(0, 1, len(preds)))

    # Treat-none baseline (always 0)
    thresholds = np.linspace(0.01, 0.70, 200)
    ax.axhline(0, color='black', ls='-', lw=1, label='Treat None')

    # Treat-all (constant line)
    first_enc = list(preds.values())[0]
    prev = first_enc['y_true'].mean()
    nb_all_line = [prev - (1 - prev) * (pt / (1 - pt)) if pt < 1 else 0 for pt in thresholds]
    ax.plot(thresholds, nb_all_line, 'k--', lw=1.5, label=f'Treat All (prev={prev:.1%})')

    for (enc_name, df), color in zip(preds.items(), colors):
        yt = df['y_true'].values.astype(float)
        yp = df['y_proba'].values.astype(float)
        ts, nb_model, _ = decision_curve(yt, yp, thresholds)
        ax.plot(ts, nb_model, color=color, lw=1.8, label=enc_name)

    ax.set_xlabel('Threshold Probability (clinician risk tolerance)', fontsize=11)
    ax.set_ylabel('Net Benefit', fontsize=11)
    ax.set_title(f'{task_name} — Decision Curve Analysis\n'
                 f'(model should be above both baselines)', fontsize=12, fontweight='bold')
    ax.set_xlim(0, 0.70)
    ax.set_ylim(-0.05, max(0.20, prev + 0.05))
    ax.legend(fontsize=8, loc='upper right')
    plt.tight_layout()
    fig.savefig(OUT_DIR / f'test8_dca_{task_name.split()[0].lower()}.png',
               dpi=160, bbox_inches='tight')
    plt.show()

print('\n✓ Test 8 complete — DCA plots saved')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 10 — Test 9: Clinical Readiness Summary (Pass/Fail Checklist)
# ═══════════════════════════════════════════════════════════════════════

print('═' * 80)
print('CLINICAL READINESS SUMMARY')
print('═' * 80)

# Define criteria
CRITERIA = {
    '1. F1 ≥ 0.80':            lambda ci_row: ci_row['F1_lo'] >= 0.75,  # lower bound near 0.80
    '2. AUC ≥ 0.90':           lambda ci_row: ci_row['AUC_lo'] >= 0.88,
    '3. Sensitivity ≥ 80%':    lambda ci_row: ci_row['Recall_lo'] >= 0.75,
    '4. Specificity ≥ 75%':    lambda ci_row: ci_row['Specificity_lo'] >= 0.70,
    '5. NPV ≥ 90%':           lambda ci_row: ci_row['NPV_lo'] >= 0.88,
    '6. CI width < 0.08':     lambda ci_row: ci_row['F1_width'] < 0.08,
}

summary_rows = []
for task_name, preds in tasks.items():
    print(f'\n{"─" * 60}')
    print(f'  {task_name}')
    print(f'{"─" * 60}')

    for enc_name in preds.keys():
        ci_row = ci_df[(ci_df['task'] == task_name) & (ci_df['encoder'] == enc_name)]
        if ci_row.empty: continue
        ci_row = ci_row.iloc[0]

        # Get calibration
        cal_row = pd.DataFrame(cal_results)
        cal_row = cal_row[(cal_row['task'] == task_name) & (cal_row['encoder'] == enc_name)]
        ece_pass = cal_row.iloc[0]['ECE_pass'] if not cal_row.empty else False

        # Get fairness
        enc_flags = [f for f in fairness_flags
                     if f['task'] == task_name and f['encoder'] == enc_name]
        fairness_pass = len(enc_flags) == 0

        print(f'\n  {enc_name}:')
        n_pass = 0
        n_total = len(CRITERIA) + 2  # +2 for calibration and fairness

        for crit_name, crit_fn in CRITERIA.items():
            passed = crit_fn(ci_row)
            n_pass += int(passed)
            symbol = '✓ PASS' if passed else '✗ FAIL'
            print(f'    {crit_name}: {symbol}')

        # Calibration
        n_pass += int(ece_pass)
        symbol = '✓ PASS' if ece_pass else '✗ FAIL'
        print(f'    7. ECE < 0.05 (calibrated): {symbol}')

        # Fairness
        n_pass += int(fairness_pass)
        symbol = '✓ PASS' if fairness_pass else f'✗ FAIL ({len(enc_flags)} violations)'
        print(f'    8. Subgroup fairness:     {symbol}')

        overall = 'CLINICALLY VIABLE ✓' if n_pass >= n_total - 1 else \
                  'CONDITIONAL ⚠' if n_pass >= n_total - 2 else \
                  'NOT READY ✗'
        print(f'    ────────────────────────────')
        print(f'    Score: {n_pass}/{n_total}  →  {overall}')

        summary_rows.append({
            'task': task_name, 'encoder': enc_name,
            'criteria_passed': n_pass, 'criteria_total': n_total,
            'verdict': overall.split()[0],
            'F1_mean': ci_row['F1_mean'], 'F1_lo': ci_row['F1_lo'],
            'AUC_mean': ci_row['AUC_mean'], 'AUC_lo': ci_row['AUC_lo'],
            'NPV_mean': ci_row['NPV_mean'], 'NPV_lo': ci_row['NPV_lo'],
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT_DIR / 'clinical_readiness_summary.csv', index=False)

# ─── Final overview ────────────────────────────────────────────
print(f'\n{"═" * 80}')
print('FINAL OVERVIEW')
print(f'{"═" * 80}')
for task_name in tasks.keys():
    task_sum = summary_df[summary_df['task'] == task_name]
    n_viable = (task_sum['verdict'] == 'CLINICALLY').sum()
    n_conditional = (task_sum['verdict'] == 'CONDITIONAL').sum()
    n_not = (task_sum['verdict'] == 'NOT').sum()
    print(f'\n  {task_name}:')
    print(f'    Clinically viable: {n_viable}/{len(task_sum)}')
    print(f'    Conditional:       {n_conditional}/{len(task_sum)}')
    print(f'    Not ready:         {n_not}/{len(task_sum)}')

    for _, row in task_sum.sort_values('criteria_passed', ascending=False).iterrows():
        print(f'      {row["encoder"]:>25s}: {row["criteria_passed"]}/{row["criteria_total"]} '
              f'F1={row["F1_mean"]:.3f}[{row["F1_lo"]:.3f}] '
              f'AUC={row["AUC_mean"]:.3f}[{row["AUC_lo"]:.3f}] '
              f'NPV={row["NPV_mean"]:.3f}[{row["NPV_lo"]:.3f}]')

print(f'\n\nAll results saved to: {OUT_DIR}')
print('═' * 80)